# Visualization Notebook

This notebook can be used for visualizing the results of Big CHUNGUS.

In [ ]:
import sys
sys.path.append('./../')

import torch
import matplotlib.pyplot as plt

from utils import *
from chungus import DINOv2FeatUp

import warnings
warnings.filterwarnings('ignore')

### Function for visualization

In [ ]:
def visualize_prediction(network, image_file, device):
    """ Visualizes a prediction
    
    :param network: network to use
    :param image_file: image file that is to be predicted
    :param device: device on which to perform prediction (must be cuda device for FeatUp)
    """

    # Read image
    image = read_image(image_file)

    # Perform inference
    network.eval()
    network.to(device)
    with torch.no_grad():
        prediction = {k: v.to('cpu') for k,v in network(image_to_tensor(image).to(device), should_normalize=True, resize=True).items()}

    # Show images
    num_images = 3 if 'reconstruction' in prediction.keys() else 2
    fig = plt.figure(figsize=(5*num_images, 4))
    axs = fig.subplots(1,num_images)
    axs[0].imshow(image)
    axs[0].axis('off')
    axs[0].set_title('Input Image')
    axs[1].imshow(prediction['prediction'], cmap='jet', vmin=0.0, vmax=1.0)
    axs[1].axis('off')
    axs[1].set_title('Traversability')
    if num_images > 2:
        axs[2].imshow(prediction['reconstruction'], cmap='gray')
        axs[2].axis('off')
        axs[2].set_title('Reconstruction / Uncertainty')
    fig.tight_layout()

### Basic settings

In [ ]:
device = torch.device('cuda') # device to use (for FeatUp, it must be a cuda device)
core_model_resolution = (224,224) # core model resolution at which inference is performed
resolution = (240,424) # resolution of images and output
model_file = './../experiments/outdoor_dinov2_224x224/best_model.pth' # model file to use

### Create and load network

In [ ]:
network = DINOv2FeatUp(core_size=core_model_resolution, output_size=resolution)
network.load_model(model_file)
network.eval()
network.to(device)
print("Model loaded")

### Visualize prediction

In [ ]:
image_file = '../../data/outdoor/images/csl_studio_2024-10-11-20-33-39_1728678917044458075.jpg'
visualize_prediction(network, image_file, device)